# Imports

In [ ]:
!pip install torch torchvision pandas regex Pillow

import os
import json
import time
import numpy as np
import pandas as pd
import regex as re
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm

import kagglehub
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

tushar5harma_path    = kagglehub.dataset_download('tushar5harma/plant-village-dataset-updated')
shubham2703_path     = kagglehub.dataset_download('shubham2703/five-crop-diseases-dataset')
zunorain_path        = kagglehub.dataset_download('zunorain/pea-plant-dataset')
mexwell_path         = kagglehub.dataset_download('mexwell/crop-diseases-classification')
ayushbpanchal_path   = kagglehub.dataset_download('ayushbpanchal/cashew-image-dataset')
tahmidmir_path        = kagglehub.dataset_download('tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh')
binnassor89_path     = kagglehub.dataset_download('binnassor89/melon-macronutrient-deficiency-dataset')
nirmalsankalana_path = kagglehub.dataset_download('nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset')
yasaminborhani_path  = kagglehub.dataset_download('yasaminborhani/wheatrustclassificationdataset')

print('Downloads complete.')


# Config

In [ ]:
# Global Settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Using device: {device}')

# Mixed precision scaler
scaler = GradScaler()

# Directories
kaggle_input = '/kaggle/input'
runs_dir     = '/kaggle/working/runs'
bank_prefix  = '/kaggle/working/runs/feature_bank'
os.makedirs(runs_dir, exist_ok=True)

base = '/kaggle/input/datasets'
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

batch_size = 64
num_workers = 4
seed = 42
n_epochs = 22
img_size = 224
early_stop_patience = 4

lr_head   = 5e-4
lr_top    = 5e-5
lr_mid    = 2e-5
lr_bottom = 1e-5

base_input = '/kaggle/input/datasets'

dataset_roots = {
    'shubham2703':     f'{base_input}/shubham2703',
    'nirmalsankalana': f'{base_input}/nirmalsankalana',
    'binnassor89':     f'{base_input}/binnassor89',
    'zunorain':        f'{base_input}/zunorain',
    'ayushbpanchal':   f'{base_input}/ayushbpanchal/cashew-image-dataset/Cashew',
    'tahmidmir':       f'{base_input}/tahmidmir',
    'yasaminborhani':  f'{base_input}/yasaminborhani',
    'mexwell':         f'{base_input}/mexwell/crop-diseases-classification/Data',
    'tushar5harma':    f'{base_input}/tushar5harma/plant-village-dataset-updated',
}


# F2 Consistency Loss Config

In [ ]:
# ============================================================
# F2 — Consistency Loss Config
# Single source of truth for LAMBDA_CONSISTENCY — used below in
# both the training-loop default arg and the per-epoch loss term.
# ============================================================

LAMBDA_CONSISTENCY      = 0.10   # sweep candidates: 0.05 / 0.10 / 0.30
f2_n_epochs              = 10     # short fine-tune run, not full training
f2_early_stop_patience   = 3

# Where your Phase C/D best checkpoints currently live
phase_ckpt_dir = '/kaggle/input/datasets/vitthaljauhari2/cropquerydataset/'

# Separate run directory so F2 checkpoints never overwrite Phase C/D
f2_runs_dir = '/kaggle/working/runs_f2'
os.makedirs(f2_runs_dir, exist_ok=True)


# Transforms

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Standard train/val transforms (used for plain CropDiseaseDataset evaluation)
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

# F2 weak/strong augmentation pair — feeds ConsistencyDataset.
# Geometric ops (crop/flip/rotate) are randomized independently per call,
# so aug_weak(img) and aug_strong(img) already differ in framing, not just color.
aug_weak = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

aug_strong = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


# Dataset Wrappers

In [ ]:
class CropDiseaseDataset(Dataset):
    """samples: list of (path_str, class_idx_int) — same format as Phase B.
    Plain single-view dataset, used for validation / before-after F1 comparison."""
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


class ConsistencyDataset(Dataset):
    """Same samples format as CropDiseaseDataset, but returns TWO augmented
    views (weak + strong) of the same image, for the F2 consistency loss."""
    def __init__(self, samples, aug_a, aug_b):
        self.samples = samples
        self.aug_a   = aug_a
        self.aug_b   = aug_b

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.aug_a(img), self.aug_b(img), label


# Label Maps

In [ ]:
dataset_label_maps = {

    # tushar5harma: keyed by (crop_dir, disease_dir) — 3-level: Crop/Split/Disease
    "tushar5harma": {
        ("Apple",        "Apple Scab")             : "Apple__Apple_Scab",
        ("Apple",        "Black Rot")              : "Apple__Black_Rot",
        ("Apple",        "Cedar Apple Rust")       : "Apple__Cedar_Apple_Rust",
        ("Apple",        "Healthy")                : "Apple__Healthy",
        ("Bell Pepper",  "Bacterial Spot")         : "Bell_Pepper__Bacterial_Spot",
        ("Bell Pepper",  "Healthy")                : "Bell_Pepper__Healthy",
        ("Cherry",       "Healthy")                : "Cherry__Healthy",
        ("Cherry",       "Powdery Mildew")         : "Cherry__Powdery_Mildew",
        ("Corn (Maize)", "Cercospora Leaf Spot")   : "Corn__Gray_Leaf_Spot",
        ("Corn (Maize)", "Common Rust")            : "Corn__Common_Rust",
        ("Corn (Maize)", "Healthy")                : "Corn__Healthy",
        ("Corn (Maize)", "Northern Leaf Blight")   : "Corn__Northern_Leaf_Blight",
        ("Grape",        "Black Rot")              : "Grape__Black_Rot",
        ("Grape",        "Esca (Black Measles)")   : "Grape__Black_Measles",
        ("Grape",        "Healthy")                : "Grape__Healthy",
        ("Grape",        "Leaf Blight")            : "Grape__Leaf_Blight",
        ("Peach",        "Bacterial Spot")         : "Peach__Bacterial_Spot",
        ("Peach",        "Healthy")                : "Peach__Healthy",
        ("Potato",       "Early Blight")           : "Potato__Early_Blight",
        ("Potato",       "Healthy")                : "Potato__Healthy",
        ("Potato",       "Late Blight")            : "Potato__Late_Blight",
        ("Strawberry",   "Healthy")                : "Strawberry__Healthy",
        ("Strawberry",   "Leaf Scorch")            : "Strawberry__Leaf_Scorch",
        ("Tomato",       "Bacterial Spot")         : "Tomato__Bacterial_Spot",
        ("Tomato",       "Early Blight")           : "Tomato__Early_Blight",
        ("Tomato",       "Healthy")                : "Tomato__Healthy",
        ("Tomato",       "Late Blight")            : "Tomato__Late_Blight",
        ("Tomato",       "Septoria Leaf Spot")     : "Tomato__Septoria_Leaf_Spot",
        ("Tomato",       "Yellow Leaf Curl Virus") : "Tomato__Yellow_Leaf_Curl_Virus",
    },

    # shubham2703: structure is Crop Diseases/Crop___Disease/{Crop}/{Class}/*.jpg
    "shubham2703": {
        "Corn___Common_Rust"          : "Corn__Common_Rust",
        "Corn___Gray_Leaf_Spot"       : "Corn__Gray_Leaf_Spot",
        "Corn___Healthy"              : "Corn__Healthy",
        "Corn___Northern_Leaf_Blight" : "Corn__Northern_Leaf_Blight",
        "Potato___Early_Blight"       : "Potato__Early_Blight",
        "Potato___Healthy"            : "Potato__Healthy",
        "Potato___Late_Blight"        : "Potato__Late_Blight",
        "Rice___Brown_Spot"           : "Rice__Brown_Spot",
        "Rice___Healthy"              : "Rice__Healthy",
        "Rice___Leaf_Blast"           : "Rice__Leaf_Blast",
        "Rice___Neck_Blast"           : "Rice__Neck_Blast",
        "Wheat___Brown_Rust"          : "Wheat__Brown_Rust",
        "Wheat___Healthy"             : "Wheat__Healthy",
        "Wheat___Yellow_Rust"         : "Wheat__Yellow_Rust",
        "Bacterial Blight"            : "Rice__Bacterial_Blight",
        "Red Rot"                     : "Sugarcane__Red_Rot",
    },

    # nirmalsankalana: only coffee classes present on disk
    "nirmalsankalana": {
        "coffee___healthy"         : "Coffee__Healthy",
        "coffee___red_spider_mite" : "Coffee__Red_Spider_Mite",
        "coffee___rust"            : "Coffee__Rust",
        "BlackMeasles"             : "Grape__Black_Measles",
        "BlackRot"                 : "Grape__Black_Rot",
        "HealthyGrapes"            : "Grape__Healthy",
        "LeafBlight"               : "Grape__Leaf_Blight",
    },

    # binnassor89: actual folder names confirmed from disk
    "binnassor89": {
        "calcium"   : "Melon__Calcium_Deficiency",
        "health"    : "Melon__Healthy",
        "nitrogen"  : "Melon__Nitrogen_Deficiency",
        "potasium"  : "Melon__Potassium_Deficiency",   # sic — matches disk
    },

    # zunorain: actual folder names confirmed from disk
    "zunorain": {
        "DOWNY_MILDEW_LEAF"  : "Pea__Downy_Mildew",
        "FRESH_LEAF"         : "Pea__Healthy",
        "LEAFMINNER_LEAF"    : "Pea__Leaf_Miner",
        "POWDER_MILDEW_LEAF" : "Pea__Powdery_Mildew",
    },

    # ayushbpanchal: train_set + test_set both present; rglob picks both
    "ayushbpanchal": {
        "anthracnose" : "Cashew__Anthracnose",
        "gumosis"     : "Cashew__Gummosis",
        "healthy"     : "Cashew__Healthy",
        "leaf miner"  : "Cashew__Leaf_Miner",
        "red rust"    : "Cashew__Red_Rust",
    },

    # tahmidmir: Original Dataset/{Class}/*.jpg — rglob parent.name = class folder
    "tahmidmir": {
        "Bacterial Leaf Spot" : "Pumpkin__Bacterial_Leaf_Spot",
        "Downy Mildew"        : "Pumpkin__Downy_Mildew",
        "Healthy Leaf"        : "Pumpkin__Healthy",
        "Mosaic Disease"      : "Pumpkin__Mosaic_Disease",
        "Powdery_Mildew"      : "Pumpkin__Powdery_Mildew",
    },

    # mexwell: CSV-based — label resolved via label_num_to_disease_map.json
    "mexwell": {
        "Cassava Bacterial Blight (CBB)"      : "Cassava__Bacterial_Blight",
        "Cassava Brown Streak Disease (CBSD)" : "Cassava__Brown_Streak_Disease",
        "Cassava Green Mottle (CGM)"          : "Cassava__Green_Mottle",
        "Cassava Mosaic Disease (CMD)"        : "Cassava__Mosaic_Disease",
        "Healthy"                             : "Cassava__Healthy",
    },
    "yasaminborhani": {
        "Brown_rust"  : "Wheat__Brown_Rust",
        "Healthy"     : "Wheat__Healthy",
        "Yellow_rust" : "Wheat__Yellow_Rust",
    },
}

# Build global class_to_idx  (must match Phase B exactly — same label maps)
all_canonical = sorted(set(
    v for mapping in dataset_label_maps.values()
    for v in mapping.values()
    if v is not None
))
class_to_idx = {cls: idx for idx, cls in enumerate(all_canonical)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
n_classes    = len(class_to_idx)
print(f"Total canonical classes: {n_classes}")


# Sample Builders

In [ ]:
def build_samples_folder(slug: str) -> list:
    label_map = dataset_label_maps[slug]
    root = Path(base) / slug
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in image_exts:
            continue
        raw_label = img_path.parent.name
        canonical = None

        # Special handling for bare 'Healthy' in shubham2703
        if slug == "shubham2703" and raw_label == "Healthy":
            crop_folder = img_path.parent.parent.name
            if crop_folder in ["Corn", "Potato", "Rice", "Wheat", "Sugarcane"]:
                canonical = f"{crop_folder}__Healthy"
                if canonical not in class_to_idx:
                    canonical = None

        if canonical is None:
            canonical = label_map.get(raw_label)

        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{slug:<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_tushar() -> list:
    label_map = dataset_label_maps["tushar5harma"]
    root = Path(base) / "tushar5harma"
    samples, skipped = [], 0
    for img_path in root.rglob("*"):
        if img_path.suffix.lower() not in image_exts:
            continue
        disease_dir = img_path.parent
        split_dir = disease_dir.parent
        crop_dir = split_dir.parent

        if disease_dir.name.startswith('.') or split_dir.name.startswith('.'):
            skipped += 1
            continue

        disease_name = disease_dir.name.strip()
        crop_name = crop_dir.name

        key = (crop_name, disease_name)
        canonical = label_map.get(key)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))
    print(f"  [{'tushar5harma':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_samples_mexwell() -> list:
    """CSV-based. Label resolved via label_num_to_disease_map.json (int key → disease name)."""
    data_root = Path(base) / "mexwell" / "crop-diseases-classification" / "Data"
    img_dir   = data_root / "train_images"
    json_path = data_root / "label_num_to_disease_map.json"
    csv_path  = data_root / "train.csv"

    with open(json_path) as f:
        num_to_name = json.load(f)   # {"0": "Cassava Bacterial Blight (CBB)", ...}

    df        = pd.read_csv(csv_path)
    label_map = dataset_label_maps["mexwell"]
    samples, skipped = [], 0

    for _, row in df.iterrows():
        img_path     = img_dir / row["image_id"]
        if not img_path.exists():
            skipped += 1
            continue
        disease_name = num_to_name[str(row["label"])]
        canonical    = label_map.get(disease_name)
        if canonical is None:
            skipped += 1
            continue
        samples.append((str(img_path), class_to_idx[canonical]))

    print(f"  [{'mexwell':<20}] {len(samples):>6} samples  ({skipped} skipped)")
    return samples


def build_all_samples():
    print("Loading datasets...")
    folder_slugs = ["shubham2703", "nirmalsankalana", "binnassor89",
                    "zunorain", "ayushbpanchal", "tahmidmir", "yasaminborhani"]
    all_samples = []
    for slug in folder_slugs:
        all_samples += build_samples_folder(slug)
    all_samples += build_samples_mexwell()
    all_samples += build_samples_tushar()
    print(f"\nTotal: {len(all_samples):,} samples | "
          f"{len(set(lbl for _, lbl in all_samples))} unique class indices")
    return all_samples


# Rebuild Train/Val Split + F2 DataLoaders

In [ ]:
all_samples = build_all_samples()

# SAME split call (test_size, random_state=seed) as the original Phase C/D
# build_dataloaders() — train_s/val_s here match the Phase C/D val set exactly,
# so F2's val_top1 is directly comparable.
labels = [lbl for _, lbl in all_samples]
train_s, val_s = train_test_split(
    all_samples, test_size=0.15, random_state=seed, stratify=labels
)
print(f"Train: {len(train_s):,} | Val: {len(val_s):,}")

f2_train_ds = ConsistencyDataset(train_s, aug_weak, aug_strong)
f2_val_ds   = CropDiseaseDataset(val_s, transform=val_transform)

kw = dict(num_workers=num_workers, prefetch_factor=4,
          persistent_workers=True, pin_memory=True)

f2_batch_size = 32   # halved vs. batch_size=64 — F2 does 2 forward passes per batch

f2_train_loader = DataLoader(f2_train_ds, batch_size=f2_batch_size, shuffle=True, **kw)
f2_val_loader   = DataLoader(f2_val_ds,   batch_size=f2_batch_size, shuffle=False, **kw)

print(f"F2 Train Batches: {len(f2_train_loader)} | F2 Val Batches: {len(f2_val_loader)}")


# Backbones

In [ ]:
class ResNet50Backbone(nn.Module):
    feature_dim = 2048

    def __init__(self, n_classes, dropout=0.4):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )

    def extract_features(self, x):
        return self.encoder(x).flatten(1)

    def forward(self, x):
        return self.head(self.extract_features(x))


class EfficientNetB3Backbone(nn.Module):
    feature_dim = 1536

    def __init__(self, n_classes, dropout=0.4):
        super().__init__()
        base = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.encoder = base.features
        self.pool = base.avgpool
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )

    def extract_features(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        return x.flatten(1)

    def forward(self, x):
        return self.head(self.extract_features(x))


class DenseNet121Backbone(nn.Module):
    feature_dim = 1024

    def __init__(self, n_classes, dropout=0.4):
        super().__init__()
        base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.encoder = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feature_dim, n_classes)
        )

    def extract_features(self, x):
        x = self.encoder(x)
        x = F.relu(x)  # no inplace
        x = self.pool(x)
        return x.flatten(1)

    def forward(self, x):
        return self.head(self.extract_features(x))


backbone_names = ['resnet50', 'efficientnet', 'densenet']

def build_all_backbones(n_classes):
    backbones = {
        'resnet50': ResNet50Backbone(n_classes).to(device),
        'efficientnet': EfficientNetB3Backbone(n_classes).to(device),
        'densenet': DenseNet121Backbone(n_classes).to(device),
    }
    for name, m in backbones.items():
        n_params = sum(p.numel() for p in m.parameters()) / 1e6
        print(f'  {name:<15} {n_params:.1f}M params')
    return backbones


# Optimizers

In [ ]:
def build_optimizer_resnet(model):
    base = model
    enc = list(base.encoder.children())
    return optim.AdamW([
        {'params': list(enc[0].parameters()) + list(enc[1].parameters()), 'lr': lr_bottom},
        {'params': enc[4].parameters(), 'lr': lr_bottom},
        {'params': enc[5].parameters(), 'lr': lr_mid},
        {'params': enc[6].parameters(), 'lr': lr_top},
        {'params': enc[7].parameters(), 'lr': lr_top},
        {'params': base.head.parameters(), 'lr': lr_head},
    ], weight_decay=1e-4)

def build_optimizer_efficientnet(model):
    base = model
    enc = list(base.encoder.children())
    n = len(enc)
    mid_idx = n // 2
    return optim.AdamW([
        {'params': enc[0].parameters(), 'lr': lr_bottom},
        {'params': sum([list(enc[i].parameters()) for i in range(1, mid_idx)], []), 'lr': lr_mid},
        {'params': sum([list(enc[i].parameters()) for i in range(mid_idx, n)], []), 'lr': lr_top},
        {'params': base.head.parameters(), 'lr': lr_head},
    ], weight_decay=1e-4)

def build_optimizer_densenet(model):
    base = model
    enc = list(base.encoder.children())
    n = len(enc)
    mid_idx = n // 2
    return optim.AdamW([
        {'params': enc[0].parameters(), 'lr': lr_bottom},
        {'params': sum([list(enc[i].parameters()) for i in range(1, mid_idx)], []), 'lr': lr_mid},
        {'params': sum([list(enc[i].parameters()) for i in range(mid_idx, n)], []), 'lr': lr_top},
        {'params': base.head.parameters(), 'lr': lr_head},
    ], weight_decay=1e-4)

def build_optimizers(backbones):
    return {
        'resnet50': build_optimizer_resnet(backbones['resnet50']),
        'efficientnet': build_optimizer_efficientnet(backbones['efficientnet']),
        'densenet': build_optimizer_densenet(backbones['densenet']),
    }


# Utilities

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.sum = 0
        self.count = 0
        self.avg = 0
    def update(self, val, n=1):
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def accuracy_topk(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        bs = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        correct = pred.t().eq(target.view(1, -1).expand_as(pred.t()))
        return [correct[:k].reshape(-1).float().sum().mul_(100.0 / bs).item() for k in topk]


@torch.no_grad()
def evaluate(name, model, loader):
    model.eval()
    t1 = t5 = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        a1, a5 = accuracy_topk(out, labels)
        bs = labels.size(0)
        t1 += a1 * bs
        t5 += a5 * bs
        total += bs
    top1 = t1 / total
    top5 = t5 / total
    print(f'  [{name}] Val top1={top1:.2f}%  top5={top5:.2f}%')
    return top1, top5


@torch.no_grad()
def ensemble_predict_loader(backbones_dict, loader):
    """Averaged-softmax ensemble prediction over a plain (img, label) loader."""
    for m in backbones_dict.values():
        m.eval()
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        probs = sum(F.softmax(m(imgs), dim=1) for m in backbones_dict.values()) / len(backbones_dict)
        preds = probs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


# Saving and Loading

In [ ]:
def save_checkpoint(name, model, optimizer, scheduler, epoch, val_top1, tag, history=None):
    path = os.path.join(runs_dir, f'{name}_{tag}.pth')
    torch.save({
        'backbone': name,
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_acc': val_top1,
        'class_to_idx': class_to_idx,
        'n_classes': n_classes,
        'history': history or [],
    }, path)
    print(f'  -> {name}_{tag}.pth  (top1={val_top1:.4f})')
    return path

def load_checkpoint(name, model, path):
    ckpt = torch.load(path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'  Loaded {name} | epoch={ckpt["epoch"]} | best_acc={ckpt["best_acc"]:.4f}')
    return ckpt


# Build Backbones, Load Phase C/D Checkpoints, Baseline (pre-F2) Eval

In [ ]:
backbones = build_all_backbones(n_classes)

# Sanity check — confirm the checkpoint files actually exist before loading
print("\nFiles in phase_ckpt_dir:")
print(os.listdir(phase_ckpt_dir))

for name in backbone_names:
    ckpt_path = os.path.join(phase_ckpt_dir, f"{name}_best.pth")
    load_checkpoint(name, backbones[name], ckpt_path)

optimizers = build_optimizers(backbones)


In [ ]:
# ============================================================
# Baseline snapshot — Phase C/D ensemble performance BEFORE F2 training.
# Must run BEFORE train_backbone_f2() below, since that will overwrite
# these same backbone weights in-place.
# ============================================================

preds_before, labels_before = ensemble_predict_loader(backbones, f2_val_loader)

report_before = classification_report(
    labels_before, preds_before,
    target_names=[idx_to_class[i] for i in range(n_classes)],
    output_dict=True, zero_division=0
)

print(f"Pre-F2 ensemble — macro F1: {report_before['macro avg']['f1-score']:.4f}  "
      f"| weighted F1: {report_before['weighted avg']['f1-score']:.4f}")


# F2 Training Loop

In [ ]:
def train_one_epoch_f2(name, model, loader, criterion, optimizer, epoch, lam=LAMBDA_CONSISTENCY):
    model.train()
    loss_meter = AverageMeter()
    ce_meter   = AverageMeter()
    kl_meter   = AverageMeter()
    top1_meter = AverageMeter()
    top5_meter = AverageMeter()
    loop = tqdm(loader, desc=f"{name} F2-E{epoch}")

    for imgs_a, imgs_b, labels in loop:
        imgs_a, imgs_b, labels = imgs_a.to(device), imgs_b.to(device), labels.to(device)

        optimizer.zero_grad()
        with autocast():
            out_a = model(imgs_a)
            out_b = model(imgs_b)

            ce_loss = criterion(out_a, labels)

            logp_a = F.log_softmax(out_a, dim=1)
            p_b = F.softmax(out_b.detach(), dim=1)   # stop-grad on the strong-aug branch's target
            kl_loss = F.kl_div(logp_a, p_b, reduction='batchmean')

            loss = ce_loss + lam * kl_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        top1, top5 = accuracy_topk(out_a, labels)
        loss_meter.update(loss.item(), imgs_a.size(0))
        ce_meter.update(ce_loss.item(), imgs_a.size(0))
        kl_meter.update(kl_loss.item(), imgs_a.size(0))
        top1_meter.update(top1, imgs_a.size(0))
        top5_meter.update(top5, imgs_a.size(0))

        loop.set_postfix(loss=loss_meter.avg, ce=ce_meter.avg, kl=kl_meter.avg, accuracy=top1_meter.avg)

    return loss_meter.avg, ce_meter.avg, kl_meter.avg, top1_meter.avg, top5_meter.avg


def train_backbone_f2(name, model, train_loader, val_loader, optimizer, n_epochs=f2_n_epochs):
    resume_path = os.path.join(f2_runs_dir, f"{name}_f2_latest.pth")
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-7)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    start_epoch = 0
    best_top1 = 0.0
    no_improve = 0
    history = []

    if os.path.exists(resume_path):
        print(f"\nResuming {name} (F2) from {resume_path}")
        ckpt = torch.load(resume_path, map_location='cpu')
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_top1 = ckpt.get('best_acc', 0.0)
        history = ckpt.get('history', [])
        print(f"  Resumed at epoch {start_epoch}, best_top1={best_top1:.4f}")

    print(f"\n{'=' * 60}")
    print(f"F2 Training: {name}  |  epochs {start_epoch}..{n_epochs - 1}  |  lambda={LAMBDA_CONSISTENCY}")
    print(f"{'=' * 60}")

    for epoch in range(start_epoch, n_epochs):
        tr_loss, tr_ce, tr_kl, tr1, tr5 = train_one_epoch_f2(
            name, model, train_loader, criterion, optimizer, epoch
        )
        val_top1, val_top5 = evaluate(name, model, val_loader)
        scheduler.step()

        history.append(dict(epoch=epoch, tr_loss=tr_loss, tr_ce=tr_ce, tr_kl=tr_kl,
                             tr_top1=tr1, val_top1=val_top1, val_top5=val_top5))
        print(f'  Epoch {epoch}: train_top1={tr1:.2f}%  val_top1={val_top1:.2f}%  '
              f'(ce={tr_ce:.4f}  kl={tr_kl:.4f})')

        # Always save "latest" so training can resume after interruption
        latest_path = os.path.join(f2_runs_dir, f'{name}_f2_latest.pth')
        torch.save({
            'backbone': name, 'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': val_top1, 'class_to_idx': class_to_idx,
            'n_classes': n_classes, 'history': history,
        }, latest_path)

        if val_top1 > best_top1:
            best_top1 = val_top1
            no_improve = 0
            best_path = os.path.join(f2_runs_dir, f'{name}_f2_best.pth')
            torch.save({
                'backbone': name, 'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_acc': val_top1, 'class_to_idx': class_to_idx,
                'n_classes': n_classes, 'history': history,
            }, best_path)
            print(f'  -> {name}_f2_best.pth  (top1={val_top1:.4f})')
        else:
            no_improve += 1
            print(f'  No improvement ({no_improve}/{f2_early_stop_patience})')
            if no_improve >= f2_early_stop_patience:
                print(f'  Early stopping {name} (F2) at epoch {epoch}')
                break

    print(f'\n{name} (F2) done. Best val top1: {best_top1:.4f}')
    return os.path.join(f2_runs_dir, f'{name}_f2_best.pth')


# Run F2 Training

In [ ]:
f2_best_paths = {}

for name in backbone_names:
    print(f"\n{'#'*60}\n# Starting {name}\n{'#'*60}")

    f2_best_paths[name] = train_backbone_f2(
        name,
        backbones[name],
        f2_train_loader,
        f2_val_loader,
        optimizers[name],
        n_epochs=f2_n_epochs
    )

    # Free this backbone + its optimizer from GPU memory before the next one starts
    backbones[name] = backbones[name].to('cpu')
    del optimizers[name]
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"  Freed {name} from GPU. Memory now: "
          f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

print("\nF2 training complete. Best checkpoints:")
for name, path in f2_best_paths.items():
    print(f"  {name}: {path}")

# Per-Class F1 — Before vs. After F2

In [ ]:
# ============================================================
# F2 — Per-class F1 delta (Before Phase C/D ensemble vs. After F2)
# `backbones` now holds the POST-F2 weights (overwritten in-place during
# training above) — `report_before` was captured earlier, before training.
# ============================================================

preds_after, labels_after = ensemble_predict_loader(backbones, f2_val_loader)

report_after = classification_report(
    labels_after, preds_after,
    target_names=[idx_to_class[i] for i in range(n_classes)],
    output_dict=True, zero_division=0
)

# Pull out the Cassava-cluster + Cashew classes for the delta that actually matters.
# Double-check this string match against the printed class names below —
# canonical labels use the "Crop__Disease" format (e.g. "Cassava__Mosaic_Disease").
weak_classes = [c for c in idx_to_class.values() if 'cassava' in c.lower() or 'cashew' in c.lower()]

print("=== F2 Per-class F1 — weak classes (before -> after) ===")
for c in weak_classes:
    f1_before = report_before.get(c, {}).get('f1-score', None)
    f1_after  = report_after.get(c, {}).get('f1-score', None)
    if f1_before is not None and f1_after is not None:
        delta = f1_after - f1_before
        sign = '+' if delta >= 0 else ''
        print(f"  {c:<35} {f1_before:.4f} -> {f1_after:.4f}   ({sign}{delta:.4f})")

print(f"\nOverall macro F1:    before={report_before['macro avg']['f1-score']:.4f}  "
      f"after={report_after['macro avg']['f1-score']:.4f}")
print(f"Overall weighted F1: before={report_before['weighted avg']['f1-score']:.4f}  "
      f"after={report_after['weighted avg']['f1-score']:.4f}")
